# 🥉 Bronze Layer Loader — Medallion Architecture

This notebook loads raw data from the `source_systems` Snowflake stage into the **Bronze schema**.

---

## Design Principles
- **No transformations** — data lands exactly as-is from the source files.
- **Incremental loading** — only new files are loaded on each run; already-processed files are skipped.
- **Dynamic tables** — tables are created from the file's own schema; nothing is pre-initialized.
- **Config-driven** — adding a new source only requires adding one entry to `SOURCE_CONFIGS`.
- **Snowpark-native** — uses Snowpark DataFrames and functions wherever possible; raw SQL only where Snowpark has no equivalent.

## Source Code Mapping
| Folder | Source System | Code |
|--------|--------------|------|
| `web_store/` | Web Store | `WEB` |
| `mobile_app/` | Mobile App | `MOB` |
| `wholesale/` | Wholesale Portal | `WHL` |
| `marketplace/` | Amazon Marketplace | `MKT` |
| `payments/` | Payment Gateway | `PAY` |
| `shipping/` | 3PL Logistics | `SHP` |
| `support/` | Customer Support | `SUP` |
| `marketing/` | Marketing | `MKTG` |
| `reviews/` | Review Aggregator | `REV` |
| `app_events/` | Mobile App Events | `APP` |

## Bronze Table Naming Convention
```
<BRONZE_SCHEMA>.<SOURCECODE>__<ENTITY>
```
Example: `BRONZE.WEB__WS_ORDERS`, `BRONZE.PAY__TRANSACTIONS`

> Double underscore `__` separates source code from entity, since Snowflake table names cannot contain dots.

## Metadata Columns Added to Every Table
| Column | Description |
|--------|-------------|
| `_SOURCE_FILE` | Stage path of the file this row came from |
| `_LOAD_TIMESTAMP` | UTC timestamp when this row was loaded |

## What Changed vs the Previous Version
| Area | Before | Now |
|------|--------|-----|
| CSV loading | `INFER_SCHEMA` + `COPY INTO` via raw SQL | `session.read` DataFrame — header detection is automatic |
| JSON loading | `COPY INTO` via raw SQL | `session.read` DataFrame |
| Incremental check | `session.sql(SELECT DISTINCT ...)` | Snowpark DataFrame `.select().distinct()` |
| Verification row counts | `session.sql(SELECT COUNT(*) ...)` | Snowpark DataFrame `.count()` |
| File format objects | Two named format objects in Snowflake | Removed entirely — options passed inline to `session.read` |
| Schema setup | `session.sql(CREATE SCHEMA ...)` | still SQL — no Snowpark equivalent |
| XML loading | `COPY INTO` via raw SQL | still SQL — Snowpark has no XML reader |
| Stage listing | `session.sql(LIST @stage)` | still SQL — no Snowpark equivalent |

In [ ]:
from datetime import datetime, timezone
from snowflake.snowpark import Session
from snowflake.snowpark.functions import lit, current_timestamp, col

session = get_active_session()

print(f"Database  : {session.get_current_database()}")
print(f"Schema    : {session.get_current_schema()}")
print(f"Warehouse : {session.get_current_warehouse()}")




---
## Cell 2 — Configuration

All settings live here. `SOURCE_CONFIGS` is the only thing you ever need to touch.

**To add a new source:** append one dict to `SOURCE_CONFIGS` — nothing else changes.

In [ ]:
# ── Environment ────────────────────────────────────────────────────────────────
BRONZE_SCHEMA = "BRONZE"          # Schema where Bronze tables live
STAGE_NAME    = "source_systems"  # Internal stage name (already exists)

# ── Source Configs ─────────────────────────────────────────────────────────────
# One entry per stage folder.
# Files inside each folder are discovered dynamically — new files are auto-detected.
#
# Keys:
#   source_code  — approved short code used as prefix in the Bronze table name
#   folder       — subfolder name inside the stage
#   file_format  — 'csv', 'json', or 'xml'

SOURCE_CONFIGS = [
    {"source_code": "WEB",  "folder": "web_store",   "file_format": "csv"},
    {"source_code": "MOB",  "folder": "mobile_app",  "file_format": "json"},
    {"source_code": "WHL",  "folder": "wholesale",   "file_format": "xml"},
    {"source_code": "MKT",  "folder": "marketplace", "file_format": "csv"},
    {"source_code": "PAY",  "folder": "payments",    "file_format": "csv"},
    {"source_code": "SHP",  "folder": "shipping",    "file_format": "xml"},
    {"source_code": "SUP",  "folder": "support",     "file_format": "csv"},
    {"source_code": "MKTG", "folder": "marketing",   "file_format": "csv"},
    {"source_code": "REV",  "folder": "reviews",     "file_format": "json"},
    {"source_code": "APP",  "folder": "app_events",  "file_format": "json"},
]

print(f"{len(SOURCE_CONFIGS)} sources configured:\n")
for cfg in SOURCE_CONFIGS:
    print(f"  {cfg['source_code']:<6}  @{STAGE_NAME}/{cfg['folder']}/  ({cfg['file_format']})")

---
## Cell 3 — One-Time Setup: Schema

Creates the Bronze schema if it does not yet exist.
Uses `session.sql` here because Snowpark has no Python API for `CREATE SCHEMA`.
Safe to re-run — `IF NOT EXISTS` makes it a no-op.

> **Note:** File format objects from the previous version have been removed entirely.
> `session.read` accepts format options inline, so named format objects in Snowflake are no longer needed.

In [ ]:
def setup_bronze_environment(session, schema):
    """Create the Bronze schema if it doesn't exist.

    SQL is used here because Snowpark does not expose a CREATE SCHEMA API.
    No file format objects are needed — session.read handles format options inline.
    """
    session.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}").collect()
    print(f"Schema '{schema}' ready.")


setup_bronze_environment(session, BRONZE_SCHEMA)

---
## Cell 4 — Helper Functions

| Function | Purpose | Snowpark or SQL? |
|----------|---------|-----------------|
| `build_table_name` | Converts a file name into the `SOURCECODE__ENTITY` Bronze table name | Pure Python |
| `list_stage_files` | Lists all files in a stage subfolder | SQL — no Snowpark equivalent for `LIST @stage` |
| `get_loaded_files` | Returns the set of files already loaded (incremental tracking) | Snowpark DataFrame |
| `load_csv` | Loads a CSV file — header detection is automatic | Snowpark `session.read` |
| `load_json` | Loads a JSON file — stores each element as a VARIANT row | Snowpark `session.read` |
| `load_xml` | Loads an XML file — stores the document as a VARIANT row | SQL — Snowpark has no XML reader |

In [ ]:
# ── Naming ─────────────────────────────────────────────────────────────────────

def build_table_name(source_code: str, file_name: str) -> str:
    """
    Derive the Bronze table name from source code and file name.
    Convention:  <SOURCECODE>__<ENTITY>   (entity = file name without extension)

    Examples:
        ("WEB",  "ws_orders.csv")      -> WEB__WS_ORDERS
        ("PAY",  "transactions.csv")   -> PAY__TRANSACTIONS
        ("MOB",  "mobile_orders.json") -> MOB__MOBILE_ORDERS
    """
    entity = file_name.rsplit(".", 1)[0].upper()
    return f"{source_code.upper()}__{entity}"


# ── Stage File Discovery ────────────────────────────────────────────────────────

def list_stage_files(session, stage_name: str, folder: str) -> list:
    """
    List all files inside @<stage_name>/<folder>/.
    Returns a list of dicts: [{stage_path, file_name}, ...].

    Uses SQL: Snowpark has no Python API equivalent for LIST @stage.
    """
    rows = session.sql(f"LIST @{stage_name}/{folder}/").collect()
    return [
        {"stage_path": row["name"], "file_name": row["name"].split("/")[-1]}
        for row in rows
    ]


# ── Incremental Tracking ────────────────────────────────────────────────────────

def get_loaded_files(session, schema: str, table_name: str) -> set:
    """
    Return the set of stage paths already present in a Bronze table.
    Reads the _SOURCE_FILE metadata column to determine what was already loaded.
    Returns an empty set if the table does not yet exist.

    Uses Snowpark DataFrame .select().distinct() instead of raw SQL.
    """
    try:
        df = session.table(f"{schema}.{table_name}")
        return {row["_SOURCE_FILE"] for row in df.select(col("_SOURCE_FILE")).distinct().collect()}
    except Exception:
        return set()  # Table does not exist yet


# ── CSV Loader ─────────────────────────────────────────────────────────────────

def load_csv(session, schema: str, stage_path: str, table_name: str):
    """
    Load a CSV file into a Bronze table using Snowpark's DataFrame reader.

    session.read automatically:
      - Reads the header row as column names  (no INFER_SCHEMA needed)
      - Infers column types                   (no manual type mapping needed)
      - Creates the target table on first load (no CREATE TABLE IF NOT EXISTS needed)

    Metadata columns _SOURCE_FILE and _LOAD_TIMESTAMP are added via
    Snowpark's with_column() before writing.
    """
    full_table = f"{schema}.{table_name}"
    stage_ref  = f"@{stage_path}"

    df = (
        session.read
        .option("header", True)                          # use row 1 as column names
        .option("infer_schema", True)                    # detect column types automatically
        .option("field_optionally_enclosed_by", '"')
        .option("null_if", "['', 'NULL', 'null']")
        .option("empty_field_as_null", True)
        .option("date_format", "AUTO")
        .option("timestamp_format", "AUTO")
        .csv(stage_ref)
    )

    # Add metadata columns using Snowpark functions
    df = (
        df
        .with_column("_SOURCE_FILE",    lit(stage_path))
        .with_column("_LOAD_TIMESTAMP", current_timestamp())
    )

    # Append to table — creates it automatically on first load
    df.write.mode("append").save_as_table(full_table)


# ── JSON Loader ────────────────────────────────────────────────────────────────

def load_json(session, schema: str, stage_path: str, table_name: str):
    """
    Load a JSON file into a Bronze table using Snowpark's DataFrame reader.

    Each top-level JSON object (or array element) is stored as one VARIANT
    row in a column called RAW_DATA. No parsing is done here — that is
    Silver's responsibility.

    strip_outer_array matches the previous COPY INTO behaviour.
    """
    full_table = f"{schema}.{table_name}"
    stage_ref  = f"@{stage_path}"

    df = (
        session.read
        .option("strip_outer_array", True)        # each array element becomes one row
        .json(stage_ref)
        .select(col("$1").alias("RAW_DATA"))       # store the full object as VARIANT
    )

    # Add metadata columns using Snowpark functions
    df = (
        df
        .with_column("_SOURCE_FILE",    lit(stage_path))
        .with_column("_LOAD_TIMESTAMP", current_timestamp())
    )

    # Append to table — creates it automatically on first load
    df.write.mode("append").save_as_table(full_table)


# ── XML Loader ─────────────────────────────────────────────────────────────────

def load_xml(session, schema: str, stage_path: str, table_name: str):
    """
    Load an XML file into a Bronze table.

    Snowpark has no XML reader — COPY INTO with TYPE=XML is the only option.
    The full document is stored as a VARIANT row. Silver will navigate it
    with XMLGET() and FLATTEN().

    Uses SQL: no Snowpark equivalent exists for XML ingestion.
    """
    full_table = f"{schema}.{table_name}"
    stage_ref  = f"@{stage_path}"

    session.sql(f"""
        CREATE TABLE IF NOT EXISTS {full_table} (
            RAW_DATA         VARIANT,
            _SOURCE_FILE     VARCHAR,
            _LOAD_TIMESTAMP  TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
        )
    """).collect()

    session.sql(f"""
        COPY INTO {full_table} (RAW_DATA, _SOURCE_FILE, _LOAD_TIMESTAMP)
        FROM (
            SELECT
                $1                         AS RAW_DATA,
                METADATA$FILENAME::VARCHAR  AS _SOURCE_FILE,
                CURRENT_TIMESTAMP()
            FROM {stage_ref}
        )
        FILE_FORMAT = (TYPE = XML)
        PURGE = FALSE
    """).collect()


print("Helper functions defined.")

---
## Cell 5 — Main Loading Function

Processes one source config entry end-to-end:
1. List all files in the stage folder.
2. Check which files are already loaded via `_SOURCE_FILE` — skip them.
3. Load each new file using the correct loader (CSV / JSON / XML).
4. Return a summary for reporting.

In [ ]:
def process_source(session, config: dict) -> dict:
    """
    Process one source config: discover files, skip already-loaded ones, load the rest.
    Returns {source, loaded, skipped, errors}.
    """
    source_code = config["source_code"]
    folder      = config["folder"]
    file_format = config["file_format"]

    print(f"\n{'─'*60}")
    print(f"Source: {source_code}  |  Folder: {folder}/  |  Format: {file_format}")

    # 1. Discover all files in the stage folder
    files = list_stage_files(session, STAGE_NAME, folder)

    if not files:
        print("  No files found in stage folder. Skipping.")
        return {"source": source_code, "loaded": 0, "skipped": 0, "errors": 0}

    print(f"  Found {len(files)} file(s).")
    loaded = skipped = errors = 0

    for file_info in files:
        stage_path = file_info["stage_path"]
        file_name  = file_info["file_name"]
        table_name = build_table_name(source_code, file_name)

        # 2. Skip files that have already been loaded (incremental check)
        already_loaded = get_loaded_files(session, BRONZE_SCHEMA, table_name)
        if stage_path in already_loaded:
            print(f"  SKIP  {file_name}  (already in {table_name})")
            skipped += 1
            continue

        # 3. Load the new file into Bronze
        try:
            print(f"  LOAD  {file_name}  ->  {BRONZE_SCHEMA}.{table_name}")

            if file_format == "csv":
                load_csv(session, BRONZE_SCHEMA, stage_path, table_name)
            elif file_format == "json":
                load_json(session, BRONZE_SCHEMA, stage_path, table_name)
            elif file_format == "xml":
                load_xml(session, BRONZE_SCHEMA, stage_path, table_name)
            else:
                raise ValueError(f"Unsupported file format: '{file_format}'")

            print(f"  Done.")
            loaded += 1

        except Exception as e:
            print(f"  ERROR loading {file_name}: {e}")
            errors += 1

    return {"source": source_code, "loaded": loaded, "skipped": skipped, "errors": errors}


print("Main loading function defined.")

---
## Cell 6 — Run

Executes the loader for every configured source and prints a final summary report.

> **Pipeline note:** This is the entry point a Snowflake Task or orchestrator should trigger.

In [ ]:
run_start = datetime.now(timezone.utc)
print(f"Bronze Load Started  {run_start.strftime('%Y-%m-%d %H:%M:%S UTC')}")
print(f"Schema : {BRONZE_SCHEMA}")
print(f"Stage  : @{STAGE_NAME}")

summary = [process_source(session, cfg) for cfg in SOURCE_CONFIGS]

# ── Summary Report ─────────────────────────────────────────────────────────────
elapsed = (datetime.now(timezone.utc) - run_start).total_seconds()

print(f"\n{'='*50}")
print(f"Finished in {elapsed:.1f}s")
print(f"{'='*50}")
print(f"{'SOURCE':<8} {'LOADED':>8} {'SKIPPED':>9} {'ERRORS':>8}")
print(f"{'─'*36}")
for r in summary:
    flag = "[!]" if r["errors"] > 0 else "[ ]"
    print(f"{flag} {r['source']:<5} {r['loaded']:>8} {r['skipped']:>9} {r['errors']:>8}")
print(f"{'─'*36}")
print(f"    {'TOTAL':<5} {sum(r['loaded']  for r in summary):>8} "
      f"{sum(r['skipped'] for r in summary):>9} "
      f"{sum(r['errors']  for r in summary):>8}")

if any(r["errors"] > 0 for r in summary):
    print("\nWARNING: Some files failed to load. Review errors above.")
else:
    print("\nAll files processed successfully.")

---
## Cell 7 — Verification

Lists all Bronze tables and their row counts as a quick sanity check after loading.
Uses Snowpark DataFrame `.count()` for row counts instead of raw `SELECT COUNT(*)` SQL.

In [ ]:
print(f"Bronze Tables in {BRONZE_SCHEMA}\n")

# SHOW TABLES still requires SQL — no Snowpark equivalent
tables = session.sql(f"SHOW TABLES IN SCHEMA {BRONZE_SCHEMA}").collect()

if not tables:
    print("No tables found. Run Cell 6 first.")
else:
    print(f"  {'TABLE NAME':<42} {'ROW COUNT':>10}")
    print("  " + "─" * 54)
    for t in tables:
        tname = t["name"]
        try:
            # Snowpark DataFrame .count() instead of SELECT COUNT(*) SQL
            count = session.table(f"{BRONZE_SCHEMA}.{tname}").count()
        except Exception:
            count = "ERROR"
        print(f"  {tname:<42} {str(count):>10}")

print("\nVerification complete.")